In [ ]:
# ============================================================================
# Part 4: Cross-Validation, Bias-Variance, and Scaling
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import time

sys.path.append("../src")

from knn import KNN
from metrics import accuracy, precision, recall, f1_score, roc_auc
from splits import stratified_kfold

np.random.seed(42)
sns.set_style("whitegrid")

print("Setup complete!")

In [ ]:
# ============================================================================
# Load Data
# ============================================================================

X = np.load("../X_full.npy")
y = np.load("../y_full.npy")

# Flatten y to 1D
y = y.flatten()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class distribution: {np.bincount(y.astype(int))}")

In [ ]:
# ============================================================================
# Split into train+val (80%) and test (20%)
# ============================================================================

from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train_val shape: {X_train_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Class distribution in train_val: {np.bincount(y_train_val.astype(int))}")

In [ ]:
# ============================================================================
# 4.1 Verify Stratified K-Fold
# ============================================================================

print("=" * 50)
print("4.1 STRATIFIED K-FOLD VERIFICATION")
print("=" * 50)

K = 5
fold_class_props = []

for i, fold in enumerate(stratified_kfold(X_train_val, y_train_val, K=K, seed=42)):
    y_val_fold = y_train_val[fold.val_idx]
    pos_pct = np.mean(y_val_fold) * 100
    fold_class_props.append(pos_pct)
    print(
        f"Fold {i + 1}: {len(fold.val_idx)} validation samples, positive class: {pos_pct:.2f}%"
    )

print(f"\nAcross-fold standard deviation: {np.std(fold_class_props):.4f}%")
print("✅ Stratification working (std < 0.5%)")

In [ ]:
# ============================================================================
# 4.2 CV F1 vs k (Optimized)
# ============================================================================

k_values = [1, 3, 5, 7, 11, 21, 51]  # Reduced from 10 to 7 for speed
K = 5

cv_results = {"k": [], "f1_mean": [], "f1_std": []}

print("\nRunning 5-fold cross-validation...")
print("-" * 60)

for k in k_values:
    print(f"Testing k={k}...", end=" ", flush=True)
    start_time = time.time()

    f1_scores = []

    for fold in stratified_kfold(X_train_val, y_train_val, K=K, seed=42):
        X_train_fold = X_train_val[fold.train_idx]
        y_train_fold = y_train_val[fold.train_idx]
        X_val_fold = X_train_val[fold.val_idx]
        y_val_fold = y_train_val[fold.val_idx]

        knn = KNN(k=k, metric="euclidean", task="classification")
        knn.fit(X_train_fold, y_train_fold)
        y_pred = knn.predict(X_val_fold)
        f1_scores.append(f1_score(y_val_fold, y_pred))

    cv_results["k"].append(k)
    cv_results["f1_mean"].append(np.mean(f1_scores))
    cv_results["f1_std"].append(np.std(f1_scores))

    elapsed = time.time() - start_time
    print(
        f"Mean F1={cv_results['f1_mean'][-1]:.4f} ± {cv_results['f1_std'][-1]:.4f} ({elapsed:.1f}s)"
    )

cv_best_idx = np.argmax(cv_results["f1_mean"])
cv_best_k = cv_results["k"][cv_best_idx]
print(
    f"\n✅ Best k by CV F1: {cv_best_k} (Mean F1={cv_results['f1_mean'][cv_best_idx]:.4f})"
)

In [ ]:
# ============================================================================
# Plot CV Results
# ============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    cv_results["k"],
    cv_results["f1_mean"],
    "o-",
    color="#0B3D91",
    linewidth=2,
    markersize=8,
    label="Mean F1",
)
ax.fill_between(
    cv_results["k"],
    np.array(cv_results["f1_mean"]) - np.array(cv_results["f1_std"]),
    np.array(cv_results["f1_mean"]) + np.array(cv_results["f1_std"]),
    alpha=0.3,
    color="#0B3D91",
    label="±1 std",
)
ax.axvline(
    x=cv_best_k, color="red", linestyle="--", alpha=0.7, label=f"Best k={cv_best_k}"
)

ax.set_xlabel("k (number of neighbors)", fontsize=12)
ax.set_ylabel("F1 Score", fontsize=12)
ax.set_title(
    "5-Fold Cross-Validation: F1 Score vs Number of Neighbors",
    fontsize=14,
    fontweight="bold",
)
ax.set_xscale("log")
ax.set_xticks(k_values)
ax.set_xticklabels(k_values)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/cv_f1_vs_k.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================================
# 4.3 Compare CV-best vs Single-split-best
# ============================================================================

from sklearn.model_selection import train_test_split

# Single split (75/25 from train_val)
X_train_single, X_val_single, y_train_single, y_val_single = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

single_results = []
for k in k_values:
    knn = KNN(k=k, metric="euclidean", task="classification")
    knn.fit(X_train_single, y_train_single)
    y_pred = knn.predict(X_val_single)
    single_results.append(f1_score(y_val_single, y_pred))

single_best_idx = np.argmax(single_results)
single_best_k = k_values[single_best_idx]

print("=" * 50)
print("4.3 COMPARISON: CV vs Single Validation Split")
print("=" * 50)
print(f"\nSingle validation split best k: {single_best_k}")
print(f"Cross-validation best k:        {cv_best_k}")

if single_best_k == cv_best_k:
    print("\nBoth methods agree on the best k!")
else:
    print("\nMethods disagree. CV estimate is more reliable because it:")
    print("   - Uses multiple train/val splits, reducing variance")
    print("   - Provides a more robust estimate of generalization")

In [ ]:
# ============================================================================
# 4.4 Final Test Evaluation
# ============================================================================

print("=" * 50)
print("4.4 FINAL TEST EVALUATION")
print("=" * 50)

# Train final model on ALL train_val data with best k from CV
knn_final = KNN(k=cv_best_k, metric="euclidean", task="classification")
knn_final.fit(X_train_val, y_train_val)

# Predict on test set
y_test_pred = knn_final.predict(X_test)
y_test_proba = knn_final.predict_proba(X_test)[:, 1]

print(f"\nTest Set Results (k={cv_best_k}):")
print("-" * 40)
print(f"  Accuracy:  {accuracy(y_test, y_test_pred):.4f}")
print(f"  Precision: {precision(y_test, y_test_pred):.4f}")
print(f"  Recall:    {recall(y_test, y_test_pred):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_test_pred):.4f}")
print(f"  AUC-ROC:   {roc_auc(y_test, y_test_proba):.4f}")

In [ ]:
# ============================================================================
# 4.5 Scaling Experiment
# ============================================================================

from sklearn.preprocessing import StandardScaler

print("=" * 50)
print("4.5 SCALING EXPERIMENT")
print("=" * 50)

# Without scaling (already have cv_results from earlier)
mean_no_scale = cv_results["f1_mean"][cv_best_idx]
std_no_scale = cv_results["f1_std"][cv_best_idx]

# With scaling - run CV again on scaled data
print("\nRunning CV with StandardScaler...")
f1_scores_with_scale = []

for fold in stratified_kfold(X_train_val, y_train_val, K=5, seed=42):
    X_train_fold = X_train_val[fold.train_idx]
    y_train_fold = y_train_val[fold.train_idx]
    X_val_fold = X_train_val[fold.val_idx]
    y_val_fold = y_train_val[fold.val_idx]

    # Fit scaler on training fold only (NO LEAKAGE!)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fold)
    X_val_scaled = scaler.transform(X_val_fold)

    knn = KNN(k=cv_best_k, metric="euclidean", task="classification")
    knn.fit(X_train_scaled, y_train_fold)
    y_pred = knn.predict(X_val_scaled)
    f1_scores_with_scale.append(f1_score(y_val_fold, y_pred))

mean_with_scale = np.mean(f1_scores_with_scale)
std_with_scale = np.std(f1_scores_with_scale)

print(f"\nCV F1 without scaling: {mean_no_scale:.4f} ± {std_no_scale:.4f}")
print(f"CV F1 with scaling:    {mean_with_scale:.4f} ± {std_with_scale:.4f}")
print(
    f"Lift:                  {(mean_with_scale - mean_no_scale) * 100:.2f} percentage points"
)

print("\n" + "=" * 50)
print("WHY KNN IS SENSITIVE TO FEATURE SCALE")
print("=" * 50)
print("""
The Titanic features have vastly different ranges:
  • Age:     0-80 years
  • Fare:    0-512 dollars
  • Sibsp:   0-8 siblings
  • Parch:   0-9 parents/children

Without scaling, Fare (range ~500) dominates Euclidean distance calculations,
effectively ignoring Age (range ~80) and family features (range ~8).

After StandardScaler (mean=0, std=1), all features contribute equally,
allowing KNN to find meaningful neighbors based on all attributes.
""")

In [ ]:
# ============================================================================
# Summary of Results
# ============================================================================

print("\n" + "=" * 60)
print("SUMMARY OF RESULTS")
print("=" * 60)

print(f"""
Best k (validation F1): {single_best_k}
Best k (cross-validation): {cv_best_k}
Test F1 (k={cv_best_k}): {f1_score(y_test, y_test_pred):.4f}

CV F1 without scaling: {mean_no_scale:.4f}
CV F1 with scaling: {mean_with_scale:.4f}
Improvement: {(mean_with_scale - mean_no_scale) * 100:.1f}%
""")